<a href="https://colab.research.google.com/github/wtryab-re/data-preprocessing/blob/main/Handling_Missing_Values.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Imports and Installs

In [1]:
!pip install -q matplotlib scikit-learn seaborn numpy pandas

In [2]:
#!/bin/bash
!kaggle datasets download krishgoyal/titanic-uncleaned

Dataset URL: https://www.kaggle.com/datasets/krishgoyal/titanic-uncleaned
License(s): unknown
100% 22.0k/22.0k [00:00<00:00, 46.9MB/s]



In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import zipfile
from sklearn.model_selection import train_test_split

In [4]:
f = zipfile.ZipFile("titanic-uncleaned.zip", "r")
f.extractall()
f.close()

In [5]:
df = pd.read_csv("titanic_train.csv")
df.shape

(891, 12)

File Overview

In [6]:
df.shape

(891, 12)

In [7]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [9]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [10]:
df.isna().sum().sort_values(ascending=False)

,0
Cabin,687
Age,177
Embarked,2
PassengerId,0
Name,0
Pclass,0
Survived,0
Sex,0
Parch,0
SibSp,0


In [11]:
missing_values = df.isna().sum().sort_values(ascending=False)
missing_percentage = (df.isna().mean()*100).sort_values(ascending=False)
pd.DataFrame({"missing_values": missing_values, "missing_percentage": missing_percentage})

,missing_values,missing_percentage
Cabin,687,77.104377
Age,177,19.865320
Embarked,2,0.224467
PassengerId,0,0.000000
Name,0,0.000000
Pclass,0,0.000000
Survived,0,0.000000
Sex,0,0.000000
Parch,0,0.000000
SibSp,0,0.000000


In [12]:
df.nunique()

,0
PassengerId,891
Survived,2
Pclass,3
Name,891
Sex,2
Age,88
SibSp,7
Parch,7
Ticket,681
Fare,248


Cabin is not continuous, it is a category, most of it is missing. 77%

Embarked is missing 2 values, could be imputed or dropped. is categorical.

Age is missing 100+ values. is continuous

drop any rows that have no target values

In [13]:
df.dropna(subset=["Survived"], inplace=True)

data leakage can also happen when values are imputed so thats something to keep in mind

In [14]:
X = df.drop(columns=["Survived"])
y = df["Survived"]

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [16]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((712, 11), (179, 11), (712,), (179,))

checking Missing Values before processing

In [17]:
missing_values = X_train.isna().sum().sort_values(ascending=False)
missing_percentage = (X_train.isna().mean()*100).sort_values(ascending=False)
pd.DataFrame({"missing_values": missing_values, "missing_percentage": missing_percentage})

,missing_values,missing_percentage
Cabin,552,77.528090
Age,137,19.241573
Embarked,2,0.280899
Name,0,0.000000
Pclass,0,0.000000
PassengerId,0,0.000000
Sex,0,0.000000
Parch,0,0.000000
SibSp,0,0.000000
Fare,0,0.000000


check for outliers and impute values based on that, if outliers then median if not then mean

dropping rows is not a good idea if there are alot of missing values because will lose alot of data

 can drop columns if the column missing is super high

In [18]:
missing_values = X_test.isna().sum().sort_values(ascending=False)
missing_percentage = (X_test.isna().mean()*100).sort_values(ascending=False)
missing_df = pd.DataFrame({"missing_values": missing_values, "missing_percentage": missing_percentage})
dropper = missing_df[missing_df["missing_percentage"]>40].index.to_list()

In [19]:
X_train.drop(columns=dropper, inplace=True)

In [20]:
X_test.drop(columns=dropper, inplace=True)

In [21]:
df.shape, X_train.shape, X_test.shape

((891, 12), (712, 10), (179, 10))

In [22]:
df.dtypes

,0
PassengerId,int64
Survived,int64
Pclass,int64
Name,object
Sex,object
Age,float64
SibSp,int64
Parch,int64
Ticket,object
Fare,float64


In [23]:
num_cols = ["Age", "SibSp", "Parch", "Fare"]
cat_cols = list(filter(lambda x: x not in num_cols, X_train.columns))
num_cols, cat_cols

(['Age', 'SibSp', 'Parch', 'Fare'],
 ['PassengerId', 'Pclass', 'Name', 'Sex', 'Ticket', 'Embarked'])

Impute values into train ONLY

In [23]:
for col in num_cols:


In [37]:
df.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [44]:
numeric_medians_of_train = {}
numeric_medians_of_test = {}
numeric_means = {}
for col in num_cols:
  numeric_medians_of_train[col] = X_train[col].median()
  numeric_medians_of_test[col] = X_test[col].median()
  numeric_means[col] = X_train[col].mean()

numeric_medians_of_train, numeric_medians_of_test, numeric_means

({'Age': 28.5, 'SibSp': 0.0, 'Parch': 0.0, 'Fare': 14.4542},
 {'Age': 27.0, 'SibSp': 0.0, 'Parch': 0.0, 'Fare': 15.2458},
 {'Age': np.float64(29.807686956521735),
  'SibSp': np.float64(0.49297752808988765),
  'Parch': np.float64(0.3904494382022472),
  'Fare': np.float64(31.819826264044945)})

In [45]:
for col in num_cols:
  X_train.fillna(numeric_medians_of_train[col], inplace=True)
  X_test.fillna(numeric_medians_of_test[col], inplace=True)

In [47]:
X_train.isna().sum()

,0
PassengerId,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0
Embarked,0


if missing in cat cols then replace missing with mode?

other ways is to implement ML to predict the label for the missing category.

can add another category called Unknown as a placeholder


In [50]:
X_train[cat_cols].isna().sum(), X_test[cat_cols].isna().sum()

(PassengerId    0
 Pclass         0
 Name           0
 Sex            0
 Ticket         0
 Embarked       0
 dtype: int64,
 PassengerId    0
 Pclass         0
 Name           0
 Sex            0
 Ticket         0
 Embarked       0
 dtype: int64)

In [54]:
for col in cat_cols:
  X_train.fillna(X_train[col].mode(), inplace=True)
  X_test.fillna(X_test[col].mode(), inplace=True)

In [56]:
X_train[cat_cols].isna().sum()

,0
PassengerId,0
Pclass,0
Name,0
Sex,0
Ticket,0
Embarked,0
